# EDA — NLI4CT
Format: JSONL  
Task: Natural Language Inference on Clinical Trials  
Metric: Accuracy  
Labels: Entailment, Contradiction, Neutral  
sentence1 = clinical trial text (long)  
sentence2 = hypothesis (short)

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid', palette='muted')

DATA_DIR = Path('../data/raw/nli4ct')

def load_jsonl(filepath):
    rows = []
    with open(filepath) as f:
        for line in f:
            rows.append(json.loads(line))
    return pd.DataFrame(rows)

# NLI4CT only has train in your folder — check what splits exist
splits = {}
for split in ['train', 'validation', 'test']:
    p = DATA_DIR / f'{split}.jsonl'
    if p.exists():
        splits[split] = load_jsonl(p)
        print(f'{split}: {len(splits[split])} examples')
    else:
        print(f'{split}: not found')

# use train as primary for EDA
df = splits.get('train', list(splits.values())[0])
print(f'\nUsing {len(df)} examples for EDA')
print('Columns:', df.columns.tolist())

## 2. Quick look

In [ ]:
df.head(5)

## 3. Label distribution

In [ ]:
label_col = 'gold_label' if 'gold_label' in df.columns else 'label'

fig, ax = plt.subplots(figsize=(7, 4))
counts = df[label_col].value_counts()
sns.barplot(x=counts.index, y=counts.values, ax=ax)
ax.set_title('Label distribution')
ax.set_ylabel('count')

for i, v in enumerate(counts.values):
    ax.text(i, v + 5, f'{100*v/len(df):.1f}%', ha='center', fontsize=11)

plt.tight_layout()
plt.show()

print(counts)
print(f'\nClass balance — majority baseline accuracy: {100*counts.max()/len(df):.1f}%')

## 4. Sentence length distribution

In [ ]:
df['len_s1'] = df['sentence1'].str.split().str.len()
df['len_s2'] = df['sentence2'].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(df['len_s1'], bins=30, ax=axes[0], kde=True)
axes[0].set_title('sentence1 length (clinical trial text)')
axes[0].set_xlabel('words')

sns.histplot(df['len_s2'], bins=30, ax=axes[1], kde=True)
axes[1].set_title('sentence2 length (hypothesis)')
axes[1].set_xlabel('words')

plt.tight_layout()
plt.show()

print('sentence1 (clinical trial context):')
print(df['len_s1'].describe().round(1))
print()
print('sentence2 (hypothesis):')
print(df['len_s2'].describe().round(1))

## 5. Length vs label — does length correlate with label?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.boxplot(data=df, x=label_col, y='len_s1', ax=axes[0])
axes[0].set_title('sentence1 length by label')

sns.boxplot(data=df, x=label_col, y='len_s2', ax=axes[1])
axes[1].set_title('sentence2 length by label')

plt.tight_layout()
plt.show()

## 6. Sample examples per label

In [ ]:
for label in df[label_col].unique():
    sample = df[df[label_col]==label].sample(1, random_state=42).iloc[0]
    print(f'--- {label} ---')
    print(f'  Hypothesis : {sample.sentence2}')
    print(f'  Context    : {sample.sentence1[:300]}...')
    print()

## 7. How we evaluate — embedding-based NLI
For embedding models we compute:
1. embed sentence1 and sentence2
2. concatenate or take difference of vectors
3. classify into Entailment / Contradiction / Neutral

Note: sentence1 is very long (clinical trial text). This will stress-test how well your embedder handles long inputs.

In [ ]:
# majority baseline
majority = df[label_col].value_counts().index[0]
majority_acc = 100 * (df[label_col] == majority).mean()
print(f'Majority class           : {majority}')
print(f'Majority baseline acc    : {majority_acc:.1f}%')
print(f'Random baseline acc      : {100/df[label_col].nunique():.1f}%')
print()
print('Your model needs to beat majority baseline to be meaningful.')
print(f'Number of classes: {df[label_col].nunique()}')

## 8. Vocabulary check — medical terms in hypotheses

In [ ]:
from collections import Counter
import re

# most common meaningful words in hypotheses (sentence2)
stopwords = {'the','a','an','is','are','was','were','of','in','to','and',
             'or','for','with','on','at','by','from','that','this','all',
             'not','do','does','did','have','has','be','been','it','its'}

words = []
for text in df['sentence2']:
    tokens = re.findall(r'[a-z]+', text.lower())
    words.extend([t for t in tokens if t not in stopwords and len(t) > 3])

top_words = Counter(words).most_common(20)
words_df = pd.DataFrame(top_words, columns=['word', 'count'])

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=words_df, x='count', y='word', ax=ax)
ax.set_title('Top 20 words in hypotheses (sentence2)')
plt.tight_layout()
plt.show()

## 9. Summary

In [ ]:
summary = {
    'total examples'          : len(df),
    'unique labels'           : df[label_col].nunique(),
    'avg sentence1 len (words)': round(df['len_s1'].mean(), 1),
    'avg sentence2 len (words)': round(df['len_s2'].mean(), 1),
    'majority baseline acc'   : f'{majority_acc:.1f}%',
    'random baseline acc'     : f"{100/df[label_col].nunique():.1f}%"
}

for k, v in summary.items():
    print(f'{k:<35}: {v}')